In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

# for the count plot and bar plot
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("US_Accidents_March23_sampled_500k.csv")
df.describe()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 46 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     500000 non-null  str    
 1   Source                 500000 non-null  str    
 2   Severity               500000 non-null  int64  
 3   Start_Time             500000 non-null  str    
 4   End_Time               500000 non-null  str    
 5   Start_Lat              500000 non-null  float64
 6   Start_Lng              500000 non-null  float64
 7   End_Lat                279623 non-null  float64
 8   End_Lng                279623 non-null  float64
 9   Distance(mi)           500000 non-null  float64
 10  Description            499999 non-null  str    
 11  Street                 499309 non-null  str    
 12  City                   499981 non-null  str    
 13  County                 500000 non-null  str    
 14  State                  500000 non-null  str    

In [3]:
# const value 
FILL_NA_VALUE = "NA"
ZERO = 0
# dropping and filling NA
dfClean = df.drop(["ID", "End_Lat", "End_Lng", "Wind_Chill(F)", "Airport_Code"], axis=1)
# fill string null with "NA"
dfClean["Description"] = dfClean["Description"].fillna(FILL_NA_VALUE)
dfClean["Street"] = dfClean["Street"].fillna(FILL_NA_VALUE)
dfClean["City"] = dfClean["City"].fillna(FILL_NA_VALUE)
dfClean["Zipcode"] = dfClean["Zipcode"].fillna(FILL_NA_VALUE)
dfClean["Timezone"] = dfClean["Timezone"].fillna(FILL_NA_VALUE)
dfClean["Weather_Timestamp"] = dfClean["Weather_Timestamp"].fillna(FILL_NA_VALUE)
dfClean["Wind_Direction"] = dfClean["Wind_Direction"].fillna(FILL_NA_VALUE)
dfClean["Weather_Condition"] = dfClean["Weather_Condition"].fillna(FILL_NA_VALUE)
dfClean["Sunrise_Sunset"] = dfClean["Sunrise_Sunset"].fillna(FILL_NA_VALUE)
dfClean["Civil_Twilight"] = dfClean["Civil_Twilight"].fillna(FILL_NA_VALUE)
dfClean["Nautical_Twilight"] = dfClean["Nautical_Twilight"].fillna(FILL_NA_VALUE)
dfClean["Astronomical_Twilight"] = dfClean["Astronomical_Twilight"].fillna(FILL_NA_VALUE)

# fill float NA with mean
dfClean["Temperature(F)"] = dfClean["Temperature(F)"].fillna(dfClean["Temperature(F)"].mean())
dfClean["Humidity(%)"] = dfClean["Humidity(%)"].fillna(dfClean["Humidity(%)"].mean())
dfClean["Pressure(in)"] = dfClean["Pressure(in)"].fillna(dfClean["Pressure(in)"].mean())
dfClean["Visibility(mi)"] = dfClean["Visibility(mi)"].fillna(dfClean["Visibility(mi)"].mean())
dfClean["Wind_Speed(mph)"] = dfClean["Wind_Speed(mph)"].fillna(dfClean["Wind_Speed(mph)"].mean())
dfClean["Precipitation(in)"] = dfClean["Precipitation(in)"].fillna(ZERO)

dfClean.info()


<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 41 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Source                 500000 non-null  str    
 1   Severity               500000 non-null  int64  
 2   Start_Time             500000 non-null  str    
 3   End_Time               500000 non-null  str    
 4   Start_Lat              500000 non-null  float64
 5   Start_Lng              500000 non-null  float64
 6   Distance(mi)           500000 non-null  float64
 7   Description            500000 non-null  str    
 8   Street                 500000 non-null  str    
 9   City                   500000 non-null  str    
 10  County                 500000 non-null  str    
 11  State                  500000 non-null  str    
 12  Zipcode                500000 non-null  str    
 13  Country                500000 non-null  str    
 14  Timezone               500000 non-null  str    

In [4]:
# Columns used to identify potential duplicate accidents, this was discovered upon realizing certain wind speed entries where duplicated 
duplicate_columns = [
    'Start_Time',
    'Start_Lat',
    'Start_Lng'
]

# Mark rows that share these values  (True = part of a duplicate group)
is_duplicate = dfClean.duplicated(
    subset=duplicate_columns,
    keep=False
)

# Use the True/False mark determine which rows are duplicates
duplicates = dfClean[is_duplicate].copy()


# Sort so each duplicate group appears together
duplicates = duplicates.sort_values(by=duplicate_columns)

# Drops duplicates in data set, only keeps first occurence of every entry found
dfClean = dfClean.drop_duplicates(subset=duplicate_columns, keep='first')
dfClean.info()

<class 'pandas.DataFrame'>
Index: 497758 entries, 0 to 499999
Data columns (total 41 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Source                 497758 non-null  str    
 1   Severity               497758 non-null  int64  
 2   Start_Time             497758 non-null  str    
 3   End_Time               497758 non-null  str    
 4   Start_Lat              497758 non-null  float64
 5   Start_Lng              497758 non-null  float64
 6   Distance(mi)           497758 non-null  float64
 7   Description            497758 non-null  str    
 8   Street                 497758 non-null  str    
 9   City                   497758 non-null  str    
 10  County                 497758 non-null  str    
 11  State                  497758 non-null  str    
 12  Zipcode                497758 non-null  str    
 13  Country                497758 non-null  str    
 14  Timezone               497758 non-null  str    
 15 

In [17]:

dfClean['Start_Time'] = pd.to_datetime(df['Start_Time'], format='mixed')
dfClean['End_Time'] = pd.to_datetime(df['End_Time'], format = 'mixed')

dfClean['Start_Time']
dfClean['End_Time']

0        2019-06-12 10:55:58
1        2022-12-04 01:56:53
2        2022-08-20 15:22:45
3        2022-02-21 19:43:23
4        2020-12-04 04:13:09
                 ...        
499995   2021-12-15 07:50:30
499996   2021-12-19 17:40:37
499997   2022-04-13 21:33:44
499998   2020-05-15 17:50:56
499999   2022-04-03 00:49:48
Name: End_Time, Length: 497758, dtype: datetime64[ns]

In [19]:

dfClean["Time_Differ"] = (
    dfClean["End_Time"] - dfClean["Start_Time"]
).dt.total_seconds() / 60

# Extract date components using the .dt accessor
dfClean["Start_Date"] = dfClean["Start_Time"].dt.date
dfClean["End_Date"] = dfClean["End_Time"].dt.date
dfClean["Year"] = dfClean["Start_Time"].dt.year
dfClean["Month"] = dfClean["Start_Time"].dt.month
dfClean["Day"] = dfClean["Start_Time"].dt.day
dfClean["Hour"] = dfClean["Start_Time"].dt.hour

# Verify data
dfClean.head()
dfClean.info()

<class 'pandas.DataFrame'>
Index: 497758 entries, 0 to 499999
Data columns (total 49 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Source                 497758 non-null  str           
 1   Severity               497758 non-null  int64         
 2   Start_Time             497758 non-null  datetime64[ns]
 3   End_Time               497758 non-null  datetime64[ns]
 4   Start_Lat              497758 non-null  float64       
 5   Start_Lng              497758 non-null  float64       
 6   Distance(mi)           497758 non-null  float64       
 7   Description            497758 non-null  str           
 8   Street                 497758 non-null  str           
 9   City                   497758 non-null  str           
 10  County                 497758 non-null  str           
 11  State                  497758 non-null  str           
 12  Zipcode                497758 non-null  str           
 13  

In [ ]:
dfDates = dfClean[["Severity","Distance(mi)", "Start_Time", "End_Time", "Time_Differ", "Start_Date", "End_Date", "Year", "Month", "Day", "Hour"]]
dfDates.info()

<class 'pandas.DataFrame'>
Index: 497758 entries, 0 to 499999
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Start_Time   497758 non-null  datetime64[ns]
 1   End_Time     497758 non-null  datetime64[ns]
 2   Time_Differ  497758 non-null  float64       
 3   Start_Date   497758 non-null  object        
 4   End_Date     497758 non-null  object        
 5   Year         497758 non-null  int32         
 6   Month        497758 non-null  int32         
 7   Day          497758 non-null  int32         
 8   Hour         497758 non-null  int32         
dtypes: datetime64[ns](2), float64(1), int32(4), object(2)
memory usage: 30.4+ MB


In [16]:
# Wind Speed Outliers Removal

# Wind speeds over 100 are unrealistic 
MAX_WIND_SPEED = 100

wind_speed_outliers = dfClean[dfClean['Wind_Speed(mph)'] > MAX_WIND_SPEED]
# Cleans dataset to only have wind speeds below 100 mph
dfClean = dfClean[(dfClean['Wind_Speed(mph)'] <= MAX_WIND_SPEED)]
print("Highest wind speed now:", dfClean['Wind_Speed(mph)'].max())


Highest wind speed now: 82.0


In [17]:
dfClean.info()

<class 'pandas.DataFrame'>
Index: 497751 entries, 0 to 499999
Data columns (total 41 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Source                 497751 non-null  str    
 1   Severity               497751 non-null  int64  
 2   Start_Time             497751 non-null  str    
 3   End_Time               497751 non-null  str    
 4   Start_Lat              497751 non-null  float64
 5   Start_Lng              497751 non-null  float64
 6   Distance(mi)           497751 non-null  float64
 7   Description            497751 non-null  str    
 8   Street                 497751 non-null  str    
 9   City                   497751 non-null  str    
 10  County                 497751 non-null  str    
 11  State                  497751 non-null  str    
 12  Zipcode                497751 non-null  str    
 13  Country                497751 non-null  str    
 14  Timezone               497751 non-null  str    
 15 